In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 

# Models from sklaearn for regression
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor  
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.neighbors import KNeighborsRegressor

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

import warnings
warnings.filterwarnings('ignore')


In [ ]:
df = pd.read_csv('./data/stud_data.csv')

In [ ]:
df.head()

# Preparing X and y

In [ ]:
X = df.drop('math score', axis = 1)
y = df['math score']

In [ ]:
X

In [ ]:
y

In [ ]:
# Create column transformer with 3 type of transformers
num_features = X.select_dtypes(exclude='object').columns
cat_features = X.select_dtypes('object').columns

In [ ]:
numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder()

In [ ]:
preprocessor = ColumnTransformer(
    [
        ('OneHotEncoder', oh_transformer, cat_features),
        ("StandardScaler", numeric_transformer, num_features)
    ]
)

In [ ]:
X = preprocessor.fit_transform(X)
X

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42)

## Creating Evaluation Function

In [ ]:
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mse)
    r2 = r2_score(true, predicted)
    return mae, mse, rmse, r2
    

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Lasso': Lasso(),
    'Ridge': Ridge(),
    "K-Nearest Neighbors": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest ": RandomForestRegressor(),
    "XGBoost": XGBRegressor(),
    "Catboost": CatBoostRegressor()    
}
model_list = []
r2_list = [] 


In [ ]:
for i in range(len(models)):
    model = list(models.values())[i]
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    model_train_mae, model_train_mse, model_train_rmse, model_train_r2 = evaluate_model(
        y_train, y_train_pred
    )
    model_test_mae, model_test_mse, model_test_rmse, model_test_r2 = evaluate_model(
        y_test, y_test_pred
    )

    model_name = list(models.keys())[i]
    print(f"Model: {model_name}\n")
    
    print(f"Model Performance for Train")
    print(f"Train MAE:  {model_train_mae:.4f}")
    print(f"Train MSE:  {model_train_mse:.4f}")
    print(f"Train RMSE: {model_train_rmse:.4f}")
    print(f"Train R2:   {model_train_r2:.4f}")

    print("-"*35)

    print(f"Model Performance for Test")

    print(f"Test MAE:   {model_test_mae:.4f}")
    print(f"Test MSE:   {model_test_mse:.4f}")
    print(f"Test RMSE:  {model_test_rmse:.4f}")
    print(f"Test R2:    {model_test_r2:.4f}")
    print("-" * 60)

    model_list.append(model_name)
    r2_list.append(model_test_r2)
    
    print("="*35)
    print("\n")

# Results

In [ ]:
model_list

In [ ]:
pd.DataFrame(list(zip(model_list, r2_list)), columns=['Model_Name', 'R2_score']).sort_values(by='R2_score' ,ascending=False)